## Getting started
First, we check the version of Tensorflow and set the seeds for reproducibility.

In [1]:
import tensorflow as tf

import numpy as np
import os
import matplotlib.pyplot as plt
import random
from tqdm import tqdm

seed = 42
img_size = (256,)*2 #gt_image size, and network output size

preprocess_data = True # turn False if you have ready your data
data_augmentation = False
BATCH_SIZE = 1

def set_seed(seedValue=42):
  """Sets the seed on multiple python modules to obtain results as
  reproducible as possible.
  Args:
  seedValue (int, optional): seed value.
  """
  np.random.seed(seed=seedValue)
  #tf.set_seed(seedValue)
  os.environ["PYTHONHASHSEED"]=str(seedValue)
  random.seed(seedValue)
set_seed(seed)
#print("Tensorflow version: ", tf.__version__ )

In [2]:
from PIL import Image

def add_padding(np_img, multiple = 256):
    '''
    Given a numpy array, add padding to the image so that the image is a multiple of 256x256
    
    Args:
      np_img: the image to be padded (uint8)
    
    Returns:
      A numpy array of the image with the padding added.
    '''

    image = Image.fromarray(np_img)
    height, width = np_img.shape

    if not width%multiple and not height%multiple:
        return np_img
    
    x = width/multiple
    y = height/multiple

    x = max(x,y)
    y = x

    new_width = int(np.ceil(x))*multiple
    new_height = int(np.ceil(y))*multiple

    left = int( (new_width - width)/2 )
    top = int( (new_height - height)/2 )
    
    result = Image.new(image.mode, (new_width, new_height), 0)
    
    result.paste(image, (left, top))

    return np.asarray(result)

In [3]:
def display(display_list, fig_size = (14,5)):
  plt.figure(figsize=fig_size)

  title = ['Image', 'Mix Image', 'Heatmap Image']

  for i in range(len(display_list)):
    plt.subplot(1, len(display_list), i+1)
    plt.title(title[i])
    plt.imshow(tf.keras.preprocessing.image.array_to_img(display_list[i]))
    #print(display_list[i]) # raw image values
    #print("---------")
    #plt.axis('off')
  plt.show()
  

In [4]:
from skimage.measure import label, regionprops

def get_just_hand(img):
    #img = label(img, connectivity=2)
    props = regionprops(img) # (min_row, min_col, max_row, max_col) -- [min; max)
    min_row, min_col, max_row, max_col = props[0]['bbox']
    return img[min_row:max_row+2, min_col-2:max_col+2]

"\nimg = np.array(Image.open(img_paths_label[0][0]))[:,:,0]<100\nplt.figure(dpi=500)\nimg = get_just_hand(img)\nplt.imshow(img, 'gray')\n"

In [5]:
def get_just_hand_256(np_img):
    
    props = regionprops(np_img) # (min_row, min_col, max_row, max_col) -- [min; max)
    min_row, min_col, max_row, max_col = props[0]['bbox']
    hand_cols = max_col - min_col
    remain = hand_cols - 256
    left = True
    N_col = np_img.shape[1]

    if remain < 0:
        # hand is smaller so, we need to add extra padding
        remain = abs(remain) # now just count remaining columns
        for i in range(remain):
            if left:
                if min_col > 0: min_col -= 1
                elif max_col+1 < N_col: max_col += 1
                else: print('ERROR: image is not big enough')
            else:
                if max_col+1 < N_col: max_col += 1
                elif min_col > 0: min_col -= 1
                else: print('ERROR')
            left = not left
    elif remain > 0:
        # hand is bigger, so we need to cut a bit more
        for i in range(remain):
            if left: min_col += 1
            else: max_col -= 1
            left = not left
            
    return np_img[:, min_col:max_col]

In [6]:
from tensorflow import keras

class Data(keras.utils.Sequence):
    """Helper to iterate over the data (as Numpy arrays)."""

    def __init__(self, batch_size, img_size, input_img_paths, labels, data_augmentation=False):
        self.batch_size = batch_size
        self.img_size = img_size
        self.input_img_paths = input_img_paths
        self.data_augmentation = data_augmentation
        self.labels = labels

    def __len__(self):
        return len(self.input_img_paths) // self.batch_size

    def get_labels(self):
        return self.labels

    def __getitem__(self, idx):
        """Returns tuple (input, target) correspond to batch #idx."""
        i = idx * self.batch_size
        batch_input_img_paths = self.input_img_paths[i : i + self.batch_size]
        batch_input_labels = self.labels[i : i + self.batch_size]

        x = np.zeros((self.batch_size,) + self.img_size + (3,), dtype="float32")
        y = np.zeros((self.batch_size,) + (1,), dtype="float32")

        for j in range(len(batch_input_img_paths)):
            x_path = batch_input_img_paths[j]
            x_image = Image.open(x_path)

            if self.data_augmentation:
                if random.random() < 0.5:
                    x_image = x_image.transpose(method=Image.FLIP_LEFT_RIGHT)
                if random.random() < 0.5:
                    x_image = x_image.transpose(method=Image.FLIP_TOP_BOTTOM)
                if random.random() < 0.5:
                    for _ in range(random.randint(1, 4)):
                        x_image = x_image.transpose(method=Image.ROTATE_90)
            
            print(x_image)
            x[j] = np.array(x_image, dtype=np.float32) / 255 # normalize x
            y[j] = batch_input_labels[j]

            x_image.close()
        return x,y

## Data modifications

### Splits

Prepare paths of input images and target segmentation masks

In [7]:
from tqdm import tqdm
from glob import glob
from sklearn.model_selection import train_test_split

input_dir = "./ScaleFree/emulated2/"

male_img_paths = glob(input_dir+"Hombres/*")
female_img_paths = glob(input_dir+"Mujeres/*")

img_paths = male_img_paths + female_img_paths
labels = [1]*len(male_img_paths) + [0]*len(female_img_paths)

img_paths_label = list(zip(img_paths, labels))

train_img_paths, tmp_img_paths = train_test_split(
    img_paths_label, 
    test_size=0.40,
    stratify= labels,
    random_state=42)

tmp_np = np.asarray(tmp_img_paths)

val_img_paths, test_img_paths = train_test_split( # mitad del 40% (tmp) en cada dataset
    tmp_img_paths, 
    test_size=0.50,
    stratify= list(tmp_np[:,1]),
    random_state=42)

# Balance classes
test_img_paths.append(val_img_paths.pop(0))

total_n_img = len(img_paths)
print("\nNumber of samples (train): \t{}  --  {}%".format( 
    len(train_img_paths), 
    round(( len(train_img_paths) * 100) / total_n_img )))
print("Number of samples (test): \t{}  --  {}%".format( 
    len(test_img_paths), 
    round(( len(test_img_paths) * 100) / total_n_img )))
print("Number of samples (val): \t{}  --  {}%".format( 
    len(val_img_paths), 
    round(( len(val_img_paths) * 100) / total_n_img )))
print("\nMale images: \t",len(male_img_paths))
print("Female images: \t",len(female_img_paths))
print(len(male_img_paths)+len(female_img_paths), "/",len(train_img_paths)+len(test_img_paths)+len(val_img_paths))


Number of samples (train): 	39  --  59%
Number of samples (test): 	15  --  23%
Number of samples (val): 	12  --  18%

Male images: 	 26
Female images: 	 40
66 / 66


### Save modified images

In [ ]:
from PIL import Image
Image.MAX_IMAGE_PIXELS = 300000000 # hay imagenes muy grandes

# data
splits = [train_img_paths, val_img_paths, test_img_paths]

# out directory names
main_Dir= "./ScaleFree/splited/emulated2/"
dirNames=[main_Dir + "train/", main_Dir + "val/", main_Dir + "test/"]
data_type = {1:"Male/", 0:'Female/'}
errors = []

if preprocess_data:

    for i, ds in enumerate(splits):
        outdir = dirNames[i]
        
        # create folders if not exist
        for folder in data_type.values():
            if not os.path.exists(outdir + folder):
                os.makedirs(outdir + folder)

        for img_path, label in tqdm(ds):
            # Get file names
            image_name = os.path.splitext(os.path.basename(img_path))[0]

            # Skip processed images
            if os.path.exists(outdir + data_type[label] + str(label) + "_" + image_name):
                continue
            
            # Open images
            try:
                image = Image.open(img_path)
            except:
                errors.append(img_path)
                continue

            # Convert (RGBA,...) images to RGB
            if image.mode != 'RGB':
                image = image.convert('RGB')

            width, height = image.size
            shorter_h = height <= width
            factor = 256/height if shorter_h else 256/width 
            new_width, new_height = round(width*factor), round(height*factor)
            image = image.resize((new_width, new_height), Image.Resampling.NEAREST) # 11k (resize equally height and width until shortest side is 256px long)

            if not shorter_h:
                image = image.transpose(Image.Transpose.ROTATE_90)

            # labeled img - cropp by bbox
            image = image.transpose(Image.Transpose.ROTATE_180) # rock
            np_img = (np.array(image)[:,:,0] < 128).astype(np.uint8) ## WARNING: Check if hand is white or black (depends the case, use: '<' or '>')
            #np_img = get_just_hand(np_img)
            if height != width:
                np_img = get_just_hand_256(np_img)  

            # Adding a padding of zeros to the image.
            #np_img = add_padding(np_img)
            np_img = np_img * 255
            np_img = np.expand_dims(np_img, axis=-1)
            np_img = np.concatenate([np_img, np_img, np_img], axis=-1)
            gt_img = Image.fromarray(np_img.astype(np.uint8))

            if not shorter_h:
                gt_img = gt_img.transpose(Image.Transpose.ROTATE_270)

            # Apply moddifications (crappify, resize, ...)
            #gt_img = gt_img.resize(img_size, Image.NEAREST)

            # Save images (without compression)
            image_name, extension = os.path.splitext(image_name)
            #gt_img.save(outdir + data_type[label] + str(label) + "_" + image_name + '.png', quality=100)
    print()
    print("Errors: ", errors)

## Check data

In [9]:
## directory names
#main_Dir= "./Data_mask/"
#dirNames=[main_Dir + "train/", main_Dir + "val/", main_Dir + "test/"]
#data_type = ["img/"]

from glob import glob
from collections import Counter

# Get paths
x_path = data_type[0]+ "*"

train_x_paths = glob(dirNames[0] + x_path)
#train_x_paths.sort()
train_labels = np.asarray([int(os.path.basename(x)[0]) for x in train_x_paths])

val_x_paths = glob(dirNames[1] + x_path)
#val_x_paths.sort()
val_labels = np.asarray([int(os.path.basename(x)[0]) for x in val_x_paths])

test_x_paths = glob(dirNames[2] + x_path)
#test_x_paths.sort()
test_labels = np.asarray([int(os.path.basename(x)[0]) for x in test_x_paths])

#test_x_paths.pop(0) # make balanced
#test_labels = test_labels[1:]

In [10]:
# data generator
train_dataset = Data( BATCH_SIZE, img_size, train_x_paths, train_labels, data_augmentation)
test_dataset = Data( BATCH_SIZE, img_size, test_x_paths, test_labels)
val_dataset = Data( BATCH_SIZE, img_size, val_x_paths, val_labels)

In [ ]:
print("train:", Counter(train_dataset.get_labels()))
print("test: ", Counter(test_dataset.get_labels()))
print("val:  ", Counter(val_dataset.get_labels()))

In [ ]:
sample_image, sample_label = train_dataset[25]
sample_image, sample_label = sample_image[0], sample_label[0] # first batch image
plt.figure(dpi=50)
plt.imshow(sample_image)
print("label: ", sample_label)

In [ ]:
sample_image[50,:,0]